In [1]:
# [0] 환경 준비
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix, classification_report, accuracy_score
from sklearn.datasets import load_diabetes, load_iris

np.random.seed(42)
plt.rcParams['figure.figsize'] = (6,4)

print("Ready!")

Ready!


In [2]:
# 회귀용: 당뇨 데이터(연속값 예측)
diabetes = load_diabetes()
X_reg, y_reg = diabetes.data, diabetes.target

# 분류용: 아이리스 데이터(다중 클래스)
iris = load_iris()
X_clf, y_clf = iris.data, iris.target

# 훈련/검증/테스트 분할(6:2:2 느낌으로 두 번 split)
X_tr_reg, X_temp_reg, y_tr_reg, y_temp_reg = train_test_split(X_reg, y_reg, test_size=0.4, random_state=42)
X_val_reg, X_te_reg, y_val_reg, y_te_reg = train_test_split(X_temp_reg, y_temp_reg, test_size=0.5, random_state=42)

X_tr_clf, X_temp_clf, y_tr_clf, y_temp_clf = train_test_split(X_clf, y_clf, test_size=0.4, random_state=42)
X_val_clf, X_te_clf, y_val_clf, y_te_clf = train_test_split(X_temp_clf, y_temp_clf, test_size=0.5, random_state=42)

X_tr_reg.shape, X_val_reg.shape, X_te_reg.shape, X_tr_clf.shape, X_val_clf.shape, X_te_clf.shape

((265, 10), (88, 10), (89, 10), (90, 4), (30, 4), (30, 4))

In [3]:
# (A) 기본 선형회귀 파이프라인: 표준화 + 선형회귀
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
]).fit(X_tr_reg, y_tr_reg)

pred_val = pipe_lr.predict(X_val_reg)
mse = mean_squared_error(y_val_reg, pred_val)
r2  = r2_score(y_val_reg, pred_val)
print(f"[Linear] Val MSE={mse:.3f}, R2={r2:.3f}")

# (B) 다항특징으로 모델 복잡도 증가 → 과적합 확인
for deg in [1, 2, 3, 4]:
    pipe_poly = Pipeline([
        ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
        ("scaler", StandardScaler(with_mean=False)), # 희소/고차시 안전
        ("lr", LinearRegression())
    ]).fit(X_tr_reg, y_tr_reg)

    mse_tr = mean_squared_error(y_tr_reg, pipe_poly.predict(X_tr_reg))
    mse_v  = mean_squared_error(y_val_reg, pipe_poly.predict(X_val_reg))
    print(f"[Poly deg={deg}] Train MSE={mse_tr:.2f} | Val MSE={mse_v:.2f}")

[Linear] Val MSE=2415.692, R2=0.581
[Poly deg=1] Train MSE=2946.85 | Val MSE=2415.69
[Poly deg=2] Train MSE=2320.38 | Val MSE=3357.75
[Poly deg=3] Train MSE=0.00 | Val MSE=16442963.28
[Poly deg=4] Train MSE=0.00 | Val MSE=167237.08


In [4]:
# (C) 릿지(Ridge)로 규제(하이퍼파라미터 α) → 과적합 완화 데모
for alpha in [0.01, 0.1, 1.0, 10.0]:
    pipe_ridge = Pipeline([
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("scaler", StandardScaler(with_mean=False)),
        ("ridge", Ridge(alpha=alpha, random_state=42))
    ]).fit(X_tr_reg, y_tr_reg)

    mse_v  = mean_squared_error(y_val_reg, pipe_ridge.predict(X_val_reg))
    print(f"[Ridge α={alpha}] Val MSE={mse_v:.2f}")

[Ridge α=0.01] Val MSE=99988.10
[Ridge α=0.1] Val MSE=25985.96
[Ridge α=1.0] Val MSE=8322.62
[Ridge α=10.0] Val MSE=4655.24


In [5]:
# (A) 소프트맥스 로지스틱 회귀(다중 클래스)
pipe_smax = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, multi_class="multinomial"))  # softmax
]).fit(X_tr_clf, y_tr_clf)

pred_val = pipe_smax.predict(X_val_clf)
proba_val = pipe_smax.predict_proba(X_val_clf)
acc = accuracy_score(y_val_clf, pred_val)
print(f"[Softmax LR] Val Accuracy={acc:.3f}")

[Softmax LR] Val Accuracy=1.000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [6]:
# (B) 혼동행렬 & 정밀도/재현율/F1 보고서
cm = confusion_matrix(y_val_clf, pred_val)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_val_clf, pred_val, target_names=iris.target_names))

Confusion Matrix:
 [[12  0  0]
 [ 0  6  0]
 [ 0  0 12]]

Classification Report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        12
  versicolor       1.00      1.00      1.00         6
   virginica       1.00      1.00      1.00        12

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [7]:
# (C) 하이퍼파라미터(C)로 규제 강도 조절 & 교차검증(간단 비교)
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=1000, multi_class="multinomial", C=C))
    ])
    cv_acc = cross_val_score(clf, X_tr_clf, y_tr_clf, cv=5, scoring="accuracy").mean()
    print(f"[C={C}] 5-fold CV Accuracy={cv_acc:.3f}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[C=0.01] 5-fold CV Accuracy=0.811


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

[C=0.1] 5-fold CV Accuracy=0.900


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

[C=1.0] 5-fold CV Accuracy=0.933


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[C=10.0] 5-fold CV Accuracy=0.922


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [8]:
# 회귀: 릿지(α=1.0 가정) 최종 선택 → 테스트 성능
best_reg = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("scaler", StandardScaler(with_mean=False)),
    ("ridge", Ridge(alpha=1.0, random_state=42))
]).fit(X_tr_reg, y_tr_reg)
pred_te_reg = best_reg.predict(X_te_reg)
print("[REG] Test MSE={:.2f}, R2={:.3f}".format(mean_squared_error(y_te_reg, pred_te_reg),
                                                r2_score(y_te_reg, pred_te_reg)))

# 분류: 소프트맥스 LR(C=1.0 가정) 최종 선택 → 테스트 성능
best_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, multi_class="multinomial", C=1.0))
]).fit(X_tr_clf, y_tr_clf)
pred_te_clf = best_clf.predict(X_te_clf)
print("[CLF] Test Accuracy={:.3f}".format(accuracy_score(y_te_clf, pred_te_clf)))
print(classification_report(y_te_clf, pred_te_clf, target_names=iris.target_names))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[REG] Test MSE=8186.29, R2=-0.415
[CLF] Test Accuracy=0.967
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        11
  versicolor       0.93      1.00      0.96        13
   virginica       1.00      0.83      0.91         6

    accuracy                           0.97        30
   macro avg       0.98      0.94      0.96        30
weighted avg       0.97      0.97      0.97        30



In [9]:
# W: (특징수 x 1), X: (샘플수 x 특징수), b: (1,)
X_small = X_tr_reg[:3, :4]              # 3개 샘플, 4개 feature만
W = np.random.randn(4, 1)               # (4,1)
b = np.random.randn(1)                  # (1,)
y_lin = X_small @ W + b                 # 행렬곱 + 편향
print("X_small shape:", X_small.shape, "| W:", W.shape, "| y_lin:", y_lin.shape)

X_small shape: (3, 4) | W: (4, 1) | y_lin: (3, 1)


In [10]:
# ================================================
# 요구사항: 회귀(degree∈{1,2,3}, alpha∈{0.1,1,10}) → Val MSE 최적 조합 선택 → Test MSE/R2
#         분류(C∈{0.01,0.1,1,10}) 5-fold CV → 최적 C 선택 → Val/Test 지표(혼동행렬·정밀도·재현율·F1)
# 전제: 아래 변수들이 이전 셀에서 준비되어 있어야 합니다.
#  - 회귀: X_tr_reg, y_tr_reg, X_val_reg, y_val_reg, X_te_reg, y_te_reg
#  - 분류: X_tr_clf, y_tr_clf, X_val_clf, y_val_clf, X_te_clf, y_te_clf
# ================================================
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold

# -----------------------------
# 과제 1 회귀 자동화
# -----------------------------
print("---과제 1 회귀 모델 최적화 시작 ---")
degrees = [1, 2, 3]
alphas  = [0.1, 1.0, 10.0]

best_mse = np.inf
best_params = {}
best_reg_model = None

for deg in degrees:
    for alpha in alphas:
        pipe_reg = Pipeline([
            ("poly",   PolynomialFeatures(degree=deg, include_bias=False)),
            ("scaler", StandardScaler()),
            ("ridge",  Ridge(alpha=alpha))
        ])
        pipe_reg.fit(X_tr_reg, y_tr_reg)
        pred_val = pipe_reg.predict(X_val_reg)
        mse_val  = mean_squared_error(y_val_reg, pred_val)
        print(f"[Val] degree={deg}, alpha={alpha} | MSE = {mse_val:.6f}")

        if mse_val < best_mse:
            best_mse = mse_val
            best_params = {"degree": deg, "alpha": alpha}
            best_reg_model = pipe_reg

print("\n---과제 1 회귀 모델 최적화 완료 ---")
print(f"최적 파라미터: {best_params} | 최적 Val MSE: {best_mse:.6f}")

# Test 평가
pred_te = best_reg_model.predict(X_te_reg)
test_mse = mean_squared_error(y_te_reg, pred_te)
test_r2  = r2_score(y_te_reg, pred_te)
print(f"[최종 평가 - 회귀] Test MSE = {test_mse:.6f}")
print(f"[최종 평가 - 회귀] Test R2  = {test_r2:.6f}")

# -----------------------------
# 과제 2 분류 자동화
# -----------------------------
print("\n\n---과제 2 분류 모델 최적화 시작 ---")
C_candidates = [0.01, 0.1, 1.0, 10.0]

# 재현성 & 라벨 분포 보존을 위한 StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_cv_acc = -1.0
best_C = None

for C_val in C_candidates:
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",  # 소프트맥스
            solver="lbfgs",
            C=C_val
        ))
    ])
    cv_acc = cross_val_score(clf, X_tr_clf, y_tr_clf, cv=cv, scoring="accuracy").mean()
    print(f"[CV 5-fold] C={C_val:<5} | Accuracy = {cv_acc:.6f}")
    if cv_acc > best_cv_acc:
        best_cv_acc = cv_acc
        best_C = C_val

print("\n---과제 2 분류 모델 최적화 완료 ---")
print(f"최적 C 값: {best_C} | 최적 CV Accuracy: {best_cv_acc:.6f}")

# 최적 C로 전체 훈련셋 재학습
best_clf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        multi_class="multinomial",
        solver="lbfgs",
        C=best_C
    ))
])
best_clf_model.fit(X_tr_clf, y_tr_clf)

# 클래스 이름(라벨) 자동 도출
try:
    # y 가 정수라면 문자열로 변환하여 보기 좋게
    unique_labels_sorted = np.unique(np.concatenate([y_tr_clf, y_val_clf, y_te_clf]))
    target_names = [str(lbl) for lbl in unique_labels_sorted]
except Exception:
    target_names = None  # 문제가 있으면 None → sklearn이 기본 라벨로 출력

# Validation 평가
print("\n---평가 1 분류 - 검증 세트 ---")
pred_val = best_clf_model.predict(X_val_clf)
print(f"Validation Accuracy: {accuracy_score(y_val_clf, pred_val):.6f}")
print("Confusion Matrix (Val):")
print(confusion_matrix(y_val_clf, pred_val, labels=unique_labels_sorted if target_names else None))
print("\nClassification Report (Val):")
print(classification_report(y_val_clf, pred_val, target_names=target_names))

# Test 평가
print("\n---평가 2 분류 - 최종 테스트 세트 ---")
pred_te = best_clf_model.predict(X_te_clf)
print(f"Test Accuracy: {accuracy_score(y_te_clf, pred_te):.6f}")
print("Confusion Matrix (Test):")
print(confusion_matrix(y_te_clf, pred_te, labels=unique_labels_sorted if target_names else None))
print("\nClassification Report (Test):")
print(classification_report(y_te_clf, pred_te, target_names=target_names))

print("\n모든 과제 단계가 완료되었습니다.")

---과제 1 회귀 모델 최적화 시작 ---
[Val] degree=1, alpha=0.1 | MSE = 2416.276966
[Val] degree=1, alpha=1.0 | MSE = 2420.960209
[Val] degree=1, alpha=10.0 | MSE = 2437.706702
[Val] degree=2, alpha=0.1 | MSE = 3184.499348
[Val] degree=2, alpha=1.0 | MSE = 2972.510160
[Val] degree=2, alpha=10.0 | MSE = 2675.618042
[Val] degree=3, alpha=0.1 | MSE = 25985.955289
[Val] degree=3, alpha=1.0 | MSE = 8322.618048
[Val] degree=3, alpha=10.0 | MSE = 4655.236990

---과제 1 회귀 모델 최적화 완료 ---
최적 파라미터: {'degree': 1, 'alpha': 0.1} | 최적 Val MSE: 2416.276966
[최종 평가 - 회귀] Test MSE = 3247.308008
[최종 평가 - 회귀] Test R2  = 0.438620


---과제 2 분류 모델 최적화 시작 ---
[CV 5-fold] C=0.01  | Accuracy = 0.833333


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

[CV 5-fold] C=0.1   | Accuracy = 0.911111
[CV 5-fold] C=1.0   | Accuracy = 0.933333
[CV 5-fold] C=10.0  | Accuracy = 0.955556

---과제 2 분류 모델 최적화 완료 ---
최적 C 값: 10.0 | 최적 CV Accuracy: 0.955556

---평가 1 분류 - 검증 세트 ---
Validation Accuracy: 1.000000
Confusion Matrix (Val):
[[12  0  0]
 [ 0  6  0]
 [ 0  0 12]]

Classification Report (Val):


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00         6
           2       1.00      1.00      1.00        12

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


---평가 2 분류 - 최종 테스트 세트 ---
Test Accuracy: 0.966667
Confusion Matrix (Test):
[[11  0  0]
 [ 0 13  0]
 [ 0  1  5]]

Classification Report (Test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11
           1       0.93      1.00      0.96        13
           2       1.00      0.83      0.91         6

    accuracy                           0.97        30
   macro avg       0.98      0.94      0.96        30
weighted avg       0.97      0.97      0.97        30


모든 과제 단계가 완료되었습니다.
